In [ ]:
# Uses 'PA Demographic Data by ZCTA.xlsx', sheet = 'Requested' from the Center for Rural PA (CRPA)
# Buils 12xZIP structure for the first time
# Merges CRPA ACS data with `12xZIP` structure

In [2]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

import pandas as pd
import numpy as np

Current Project Root Directory set to: /Users/luca/Documents/GitHub/US_conflict_forecasting
Current Search Path set to: ['/Users/luca/Documents/GitHub/US_conflict_forecasting', '/Users/luca/Documents/GitHub/US_conflict_forecasting/1_Features/d_Sociodemographic', '/Users/luca/anaconda3/envs/ML_Lab/lib/python312.zip', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/lib-dynload', '', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages', '/Users/luca/Documents/GitHub/US_conflict_forecasting']


In [4]:
#Read demographic data from center
file_path = os.path.join(project_root_dir, '2_Dataset/a_Input_Datasets/PA Demographic Data by ZCTA - Copy.xlsx')
sheet_name = 'Transposed Data'

# Read the Excel file
data = pd.read_excel(file_path, sheet_name=sheet_name)

In [5]:
# x12 each ZIP code to have 12 months, by creating a new df that has 12 rows for each ZIP code
monthly_ZCTA = pd.DataFrame()

months = pd.date_range(start='2022-01-01', end='2022-12-01', freq='MS').strftime('%Y-%m').tolist()

for index, row in data.iterrows():
    zip_code = row['ZCTA']
    zip_code_entries = pd.DataFrame({'ZCTA': [zip_code] * len(months), 'Month': months})
    monthly_ZCTA = pd.concat([monthly_ZCTA, zip_code_entries])

In [6]:
# Merge the two dataframes
final_12_zip_ACS = pd.merge(monthly_ZCTA, data, on = 'ZCTA', how = 'left')

In [7]:
# Drop NAs in ZIP code column
final_12_zip_ACS = final_12_zip_ACS.dropna(subset=['ZCTA'])

In [9]:
print(data.columns)

Index(['ZCTA', 'Name of ZIP Code', 'Population, 2020',
       '# Square Land Miles, 2020', 'Population per Square Land Mile, 2022',
       'Rural_Urban_Commerical Status', 'Total Population, 2022',
       'Males, 2022', 'Females, 2022', '<18 Years Old, 2022',
       ...
       'Homeowners with Mortgage Whose Housing Costs was Calculated, 2022',
       'Homeowners with Mortgage Paying <30% Income for Housing Cost, 2022',
       'Homeowners with Mortgage Paying 30% to 49%  Income for Housing Cost, 2022',
       'Homeowners with Mortgage Paying 50%+  Income for Housing Cost, 2022',
       '# Households Paying Cash Rent', 'Median Gross Rent, 2022',
       '# Renters for Whom Income to Rent Ratio was Determined, 2022',
       '<30% of Income for Rent', '30% to 49% of Income for Rent, 2022',
       '50%+ of Income for Rent, 2022'],
      dtype='object', length=173)


In [10]:
# Additional columns that hold NAs, essentially white space columns in og excel file
na_columns = final_12_zip_ACS.columns[final_12_zip_ACS.isna().any()].tolist()

# Only specify white space columns. (This was for the requested data tab)
#columns_to_drop = ['Unnamed: 6', 'Unnamed: 9', 'Unnamed: 14', 'Unnamed: 21', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'ZCTA.1', 'X', 'Y', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37']

# Drop specified columns.
#final_12_zip_ACS = final_12_zip_ACS.drop(columns=columns_to_drop)

In [11]:
#Ensure that datasets are all in the same order + add index
#Sorting all ZIPs in ascending order 
final_12_zip_ACS = final_12_zip_ACS.sort_values(by=['ZCTA', 'Month'], ascending=[True, True])

#Resetting index 
final_12_zip_ACS.reset_index(drop=True, inplace=True)

In [12]:
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_1 = "All_SocDem_Features.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

final_12_zip_ACS.to_csv(output_path_1, index=False)